In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import shutil

In [2]:
def compute_Psat_w(T):
    """
    Returns water liquid saturation pressure in Pascal.

    Parameters
    ----------
    T : Union[float, np.ndarray]
        Temperature in Kelvin.

    Returns
    -------
    Union[float, np.ndarray]
        H2O liquid saturation pressure in Pascal.
    """
    return 100.0 * np.exp( - 6096.9385 / T \
                            + 16.635794 \
                            - 0.02711193 * T \
                            + 1.673952E-5 * T * T \
                            + 2.433502 * np.log( T ) 
    )

def compute_Psat_i(T):
    """
    Returns water solid saturation pressure in Pascal.

    Parameters
    ----------
    T : Union[float, np.ndarray]
        Temperature in Kelvin.

    Returns
    -------
    Union[float, np.ndarray]
        H2O solid saturation pressure in Pascal.
    """
    return 100.0 * np.exp( - 6024.5282 / T \
                        + 24.7219 \
                        + 0.010613868 * T \
                        - 1.3198825E-5 * T * T \
                        - 0.49382577 * np.log( T ) )

In [3]:
test_num = 2

# Define the file paths
input_file_path = '/home/chinahg/GCresearch/contrailuncertainty/PCE/0093_20231012_2250_238.nc'
output_file_template = '/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/test_{}/APCEMM_input_run_{}.nc'

# Open the input NetCDF file
ds = xr.open_dataset(input_file_path)

/home/chinahg/.conda/envs/contrails/lib/python3.9/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.30.0
  warnings.warn(


In [15]:
# Make this applicable to PCE by updating with sampled values and saving input files
# Start with 10 training examples
mean_Y = 0
sigma_Y = 0.5
altitudes = 21
timesteps = 7

scaling = 15
scaled_mean = 120

# Sample fluctuations in RH values
RHi_sampled = np.random.normal(mean_Y, sigma_Y, size = timesteps) * scaling + scaled_mean
T = 210 #np.array(ds['temperature'].isel(altitude = 12))

# # Calculate RHw from RHi
# P_sat_w = compute_Psat_w(T) # [Pa] Bolton 1980
# P_sat_i =  compute_Psat_i(T) # [Pa] Guide to Meteorological Instruments and Methods of Observation (CIMO Guide) (WMO, 2008)
        
# RH_w = (RHi_sampled*P_sat_i)/P_sat_w # [%] Relative humidity wrt ice from relative humidity wrt water
# print(P_sat_i/P_sat_w)

# print(RH_w)
print(RHi_sampled)
# Check if RH_sampled has any zero or negative values
if np.any(RHi_sampled <= 0):
    print("RHi_sampled contains zero or negative values.")
else:
    print("RHi_sampled does not contain any zero or negative values.")

[113.99603869 120.75717546 114.55322882 118.01705459 111.80042935
 113.60479265 107.4984272 ]
RHi_sampled does not contain any zero or negative values.


In [16]:
# Replace the 18th row of ds with the 18th row of values in RH_sampled
ds['relative_humidity_ice'][12, :] = RHi_sampled
# ds['relative_humidity'][12, :] = RH_w
ds['temperature'][12, :] = 216.0 #[K]

# Save the changes to new output files
output_file_path = output_file_template.format(test_num,test_num)
ds.to_netcdf(output_file_path)

ds.close()

In [5]:
ds = xr.open_dataset('/home/chinahg/GCresearch/contrailuncertainty/PCE/APCEMM_training_sets/test_2/APCEMM_input_run_2.nc')
df = ds.to_dataframe()
ds

<xarray.Dataset>
Dimensions:                (altitude: 21, time: 7)
Coordinates:
  * altitude               (altitude) float64 4.206 4.53 4.865 ... 14.56 15.8
  * time                   (time) int64 22 23 0 1 2 3 4
Data variables:
    var_length             int64 21
    pressure               (altitude) float64 600.0 575.0 550.0 ... 125.0 100.0
    temperature            (altitude, time) float64 268.8 268.6 ... 214.4 214.5
    relative_humidity      (altitude, time) float64 32.73 34.88 ... 1.716 1.668
    relative_humidity_ice  (altitude, time) float64 34.16 36.45 ... 3.058 2.969
    shear                  (altitude, time) float64 -0.001595 ... -0.00113
    w                      (altitude, time) float64 0.04489 ... -0.02519
    hrrr_forecast_time     datetime64[ns] 2023-10-12T12:00:00

In [18]:
ds['temperature'].isel(altitude = 12)

<xarray.DataArray 'temperature' (time: 7)>
array([216., 216., 216., 216., 216., 216., 216.])
Coordinates:
    altitude  float64 9.164
  * time      (time) int64 22 23 0 1 2 3 4

In [19]:
ds.close()